In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.Newton(model=model, lr_init=1e-3)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.677232563495636
epoch:  1, loss: 0.6654592752456665
epoch:  2, loss: 0.653428316116333
epoch:  3, loss: 0.6413272023200989
epoch:  4, loss: 0.6301077008247375
epoch:  5, loss: 0.6190567016601562
epoch:  6, loss: 0.608486533164978
epoch:  7, loss: 0.5961899161338806
epoch:  8, loss: 0.5831252336502075
epoch:  9, loss: 0.57127445936203
epoch:  10, loss: 0.559428870677948
epoch:  11, loss: 0.5481939315795898
epoch:  12, loss: 0.5372581481933594
epoch:  13, loss: 0.5265744924545288
epoch:  14, loss: 0.5161817073822021
epoch:  15, loss: 0.5059760808944702
epoch:  16, loss: 0.49610793590545654
epoch:  17, loss: 0.4864376485347748
epoch:  18, loss: 0.4769810438156128
epoch:  19, loss: 0.4678351581096649
epoch:  20, loss: 0.45887869596481323
epoch:  21, loss: 0.4500712454319
epoch:  22, loss: 0.4413292109966278
epoch:  23, loss: 0.4328744411468506
epoch:  24, loss: 0.4246272146701813
epoch:  25, loss: 0.4164240062236786
epoch:  26, loss: 0.4084445536136627
epoch:  27, loss: 

In [7]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8833572864532471
Test metrics:  R2 = 0.8088124394416809


In [8]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.4069918096065521
epoch:  1, loss: 0.30078428983688354
epoch:  2, loss: 0.23941805958747864
epoch:  3, loss: 0.19068877398967743
epoch:  4, loss: 0.12384755909442902
epoch:  5, loss: 0.11000995337963104
epoch:  6, loss: 0.09744400531053543
epoch:  7, loss: 0.06534117460250854
epoch:  8, loss: 0.018194397911429405
epoch:  9, loss: 0.016934653744101524
epoch:  10, loss: 0.01692317984998226
epoch:  11, loss: 0.01588089019060135
epoch:  12, loss: 0.00923912413418293
epoch:  13, loss: 0.006572662852704525
epoch:  14, loss: 0.005165842827409506
epoch:  15, loss: 0.004390491638332605
epoch:  16, loss: 0.00426678778603673
epoch:  17, loss: 0.0034941588528454304
epoch:  18, loss: 0.003163896733894944
epoch:  19, loss: 0.002895803190767765
epoch:  20, loss: 0.0026518823578953743
epoch:  21, loss: 0.0024378171656280756
epoch:  22, loss: 0.0022565380204468966
epoch:  23, loss: 0.0021044486202299595
epoch:  24, loss: 0.0019767761696130037
epoch:  25, loss: 0.001863095909357071
epo

In [9]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9987500905990601
Test metrics:  R2 = 0.9968540668487549
